# The first model

The last notebook turned every map into 39 numbers, all of them known before the map is
played. This one fits a model to those numbers and finds out whether it can predict the
winner any better than guessing.

**What we expect.** Notebook 03 found that no single agent's presence predicts winning
beyond chance. So a model built on agent picks alone should land close to a coin flip.
That is the honest expected result, not a failure — and if it instead scores very highly,
that means we have made a mistake somewhere and should go looking for it.

**What we compare against.** Two deliberately dumb predictions:

| Guess | How often it is right |
|---|---|
| Always say team A wins | 51.1% |
| Guess whoever picked the map, and team A when nobody picked it | 54.6% |

Beating 50% is not enough. The model has to beat 54.6% to have earned anything.

**Why the second one has that awkward second half.** Notebook 02 reported that the team
who picked a map wins it 53.8% of the time, and that figure is correct — but it only
describes maps somebody actually picked. The last map of a match is whatever is left after
both teams have banned, so nobody picks it. That is **225 of the 1,272 maps, 18% of the
season**, and on those the rule has no opinion at all.

A model has to produce an answer for every map, so a rule it is measured against has to do
the same. Giving the rule a fallback — guess team A on decider maps — makes it answer
everywhere, and that complete rule is right **54.6%** of the time. Comparing the model
against the 53.8% figure instead would be comparing a score over all maps with a score over
the 82% of maps the rule happens to have an opinion about.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

from dataset import build_map_table
from features import build_features

maps = build_map_table()
X, y = build_features(maps)

print(f"{len(maps):,} maps, each described by {X.shape[1]} numbers")

1,272 maps, each described by 39 numbers


## 1. Splitting the season into three, not two

So far we have split the season in two: learn from everything up to Masters Toronto, test
on what came after. That is enough to score one model, but not enough once we start trying
things out.

The problem is that from tomorrow we will be comparing options — different ways of
describing the data, different kinds of model. If we check every attempt against the test
set and keep whichever scores best, we are no longer testing the model on matches it has
never seen. We are choosing the model that happens to suit those particular matches. The
final number would look better than the truth.

So we split three ways instead:

| Part | What it is for |
|---|---|
| **Train** | what the model learns from |
| **Validation** | where we compare attempts against each other |
| **Test** | untouched until we have committed to a final model |

The test half stays exactly as it was. The question is where to cut the validation set out
of the training half — and to answer that we need to see how the season is laid out.

In [2]:
from dataset import SPLIT_DATE, split_season

learn, test = split_season(maps)

timeline = (
    maps.groupby("tournament")
    .agg(first_map=("played_at", "min"), last_map=("played_at", "max"), maps=("map", "size"))
    .sort_values("first_map")
)
timeline["half"] = np.where(timeline["first_map"] < SPLIT_DATE, "learn from", "test on")

print(timeline.to_string(formatters={
    "first_map": lambda d: d.strftime("%d %b"),
    "last_map": lambda d: d.strftime("%d %b"),
}))

print(f"\nsplit at {SPLIT_DATE:%d %b %Y}")
print(f"learn from: {len(learn):,} maps      test on: {len(test):,} maps")
print(f"tournaments split across both halves: {len(set(learn['tournament']) & set(test['tournament']))}")

                              first_map last_map  maps        half
tournament                                                        
VCT 2025: China Kickoff          11 Jan   25 Jan    56  learn from
VCT 2025: EMEA Kickoff           15 Jan   09 Feb    60  learn from
VCT 2025: Americas Kickoff       17 Jan   09 Feb    56  learn from
VCT 2025: Pacific Kickoff        18 Jan   09 Feb    63  learn from
Valorant Masters Bangkok 2025    20 Feb   02 Mar    42  learn from
VCT 2025: China Stage 1          13 Mar   04 May   105  learn from
VCT 2025: Americas Stage 1       22 Mar   05 May   104  learn from
VCT 2025: Pacific Stage 1        22 Mar   11 May   107  learn from
VCT 2025: EMEA Stage 1           26 Mar   18 May   104  learn from
Valorant Masters Toronto 2025    07 Jun   22 Jun    59  learn from
VCT 2025: China Stage 2          03 Jul   24 Aug   110     test on
VCT 2025: Pacific Stage 2        15 Jul   31 Aug   103     test on
VCT 2025: EMEA Stage 2           16 Jul   31 Aug   110     tes

## 2. Where the validation set goes

The test half is now safely set aside. That leaves 756 maps to both learn from *and*
compare attempts on, and we have to carve a validation set out of them somehow.

The obvious move is to slice off the last chunk of those 756 and call it validation. The
trouble is that the only clean gap in the season leaves about 59 maps. A score measured on
59 maps carries a margin of error of roughly **±13 percentage points** — so a model that
truly sits at 52% could easily read as 65%, or as 39%. We would be picking between options
using a ruler that cannot tell them apart.

So instead we do it five times over, and average. Take the matches in date order, cut them
into six chunks, and each time train on everything before a chunk and check against that
chunk:

| Fold | Learns from | Checked against |
|---|---|---|
| 1 | chunk 1 | chunk 2 |
| 2 | chunks 1–2 | chunk 3 |
| 3 | chunks 1–3 | chunk 4 |
| 4 | chunks 1–4 | chunk 5 |
| 5 | chunks 1–5 | chunk 6 |

Every fold still trains on the past and is checked on the future, which is the whole point
of splitting by date in the first place. But now the score is an average over five
different checks covering 629 maps in total rather than 59.

How much that actually buys us needs stating carefully. If those 629 maps were 629
independent results, the margin would be about **±4 points**. They are not: they come from
only **246 matches**, and in **53% of matches one team wins every map**, so maps from the
same match largely repeat each other. The honest figure sits between **±4 and ±6 points**.

That is still far better than the single chunk — whose ±13 was itself optimistic for the
same reason, since 59 maps is only about 23 matches. But it is not tight enough to treat a
two-point difference between two models as real. What it *is* good enough for is spotting a
model that is consistently worse across every fold, which is what happens below.

The training set grows as the folds go on, rather than sliding along at a fixed size. A
sliding window would make later folds forget January, which is arguably right given how
much the game changes across a season — but with only 756 maps to begin with, throwing
data away costs more than the staleness does. Worth revisiting if we ever add more seasons.

**One catch that matters.** Our rows are *maps*, not matches, and a match is 2–5 maps. If
we cut the 756 rows into six chunks by position, matches get sliced in half — map 1 of a
match lands in training, map 3 in validation. That would flatter us: the model would be
half-remembering a match rather than facing a fresh one, because whichever team is playing
better that night tends to take several maps. So the cut is made **between matches**, never
inside one.

In [3]:
# season_folds lives in src/dataset.py so notebook 05 uses exactly the
# same splitter rather than its own hand-copied version.
from dataset import N_FOLDS, season_folds

summary = []
for fold, (learn_rows, check_rows) in enumerate(season_folds(learn), start=1):
    a, b = learn.loc[learn_rows], learn.loc[check_rows]
    summary.append({
        "fold": fold,
        "learn maps": len(a),
        "learn matches": a["match_id"].nunique(),
        "check maps": len(b),
        "check matches": b["match_id"].nunique(),
        "checked on": f"{b['played_at'].min():%d %b} - {b['played_at'].max():%d %b}",
        "overlap": len(set(a["match_id"]) & set(b["match_id"])),
    })

print(pd.DataFrame(summary).to_string(index=False))
print(f"\ntotal maps used for checking: {sum(row['check maps'] for row in summary):,}")

 fold  learn maps  learn matches  check maps  check matches      checked on  overlap
    1         127             50         136             50 25 Jan - 27 Feb        0
    2         263            100         124             49 28 Feb - 31 Mar        0
    3         387            149         122             49 31 Mar - 14 Apr        0
    4         509            198         116             49 14 Apr - 03 May        0
    5         625            247         131             49 03 May - 22 Jun        0



total maps used for checking: 629


## 3. The first model

Logistic regression, left on its settings as they come. Nothing is tuned yet — the point of
this run is to get one honest number on the board to measure later attempts against. Tuning
first would leave us unable to say whether a change helped or whether we simply got a
kinder set of folds.

Worth being precise about one thing: "as it comes" is not the same as "unrestrained".
`LogisticRegression` arrives with an L2 penalty already applied at middling strength
(`C=1.0`), which holds the coefficients back from chasing noise. So when we tune `C`
tomorrow we are not switching that on, we are moving it up or down from where it already
sits.

Both baselines are recalculated **on each fold's own validation maps** rather than reused
from the season-wide figures. Each chunk is only ~125 maps, so its own "team A wins" rate
drifts from the season's 51%, and comparing a fold score against a season number would be
comparing two different things.

In [4]:
from sklearn.linear_model import LogisticRegression

X_learn, y_learn = build_features(learn)

scores = []
for fold, (learn_rows, check_rows) in enumerate(season_folds(learn), start=1):
    model = LogisticRegression(max_iter=1000)
    model.fit(X_learn.loc[learn_rows], y_learn.loc[learn_rows])

    truth = y_learn.loc[check_rows]
    checked = learn.loc[check_rows]

    # Baselines measured on these same maps, not borrowed from the whole season.
    always_a = (truth == 1).mean()

    # The picker rule has to answer on every map the model answers on, so it
    # falls back to "team A" on decider maps, where nobody picked and the rule
    # has no opinion. Without the fallback it would be scored on fewer maps
    # than the model, which is not a fair comparison.
    picked_by_a = checked["map_picked_by"].map({"team_a": 1, "team_b": 0})
    picker_rule = (picked_by_a.fillna(1) == truth).mean()

    scores.append({
        "fold": fold,
        "maps": len(truth),
        "deciders": int(picked_by_a.isna().sum()),
        "always A": always_a,
        "map picker": picker_rule,
        "model": model.score(X_learn.loc[check_rows], truth),
    })

results = pd.DataFrame(scores).set_index("fold")

as_percent = results.copy()
for column in ("always A", "map picker", "model"):
    as_percent[column] = (as_percent[column] * 100).round(1)
print(as_percent.to_string())
print(f"\nthe model was fitted on {X_learn.shape[1]} columns")
print("\naverage across folds:")
for column in ("always A", "map picker", "model"):
    print(f"  {column:<12} {results[column].mean() * 100:5.1f}%   "
          f"(fold range {results[column].min() * 100:.1f} - {results[column].max() * 100:.1f})")

      maps  deciders  always A  map picker  model
fold                                             
1      136        23      52.9        58.8   53.7
2      124        22      54.8        58.9   52.4
3      122        24      50.0        50.8   50.0
4      116        18      45.7        50.0   53.4
5      131        16      48.9        58.8   47.3

the model was fitted on 38 columns

average across folds:
  always A      50.5%   (fold range 45.7 - 54.8)
  map picker    55.5%   (fold range 50.0 - 58.9)
  model         51.4%   (fold range 47.3 - 53.7)


## 4. Why does it lose to the baseline?

The model scores 51.4%. The picker rule scores 55.5% on the same folds. Losing to a rule
that fits in one sentence needs explaining, because there is a version of this that is a
bug and a version that is a finding, and the two need telling apart.

Notice what the model was handed. Of its 38 columns:

- **27** describe which agents each side picked — and notebook 03 already showed that no
  single agent's presence predicts winning beyond chance.
- **10** say which map was played.
- **1**, `map_picked_by_a`, is the column the picker rule is built out of.

*(38 and not the 39 printed at the top: `build_features` derives its map columns from
whatever maps appear in the data it is given, and Corrode was not played until July, so it
is absent from the learning half. That is convenient here and a problem later — see the
note at the end.)*

The penalty that holds the coefficients back applies to all 39 equally. It has no way of
knowing that one of them is the only one worth listening to. So the suspicion is that the
one useful column is being drowned out by 38 that are mostly noise.

That is testable. Fit the same model on each group of columns on its own and see which
groups actually help.

In [5]:
agent_columns = [c for c in X_learn.columns if c.startswith("agent_")]
map_columns = [c for c in X_learn.columns if c.startswith("map_") and c != "map_picked_by_a"]
picker_column = ["map_picked_by_a"]

# Column counts are worked out rather than written in, so the labels cannot
# drift away from what was actually fitted.
groups = {
    name: columns
    for name, columns in [
        (f"picker only ({len(picker_column)})", picker_column),
        (f"picker + map ({len(picker_column + map_columns)})", picker_column + map_columns),
        (f"agents only ({len(agent_columns)})", agent_columns),
        (f"agents + map ({len(agent_columns + map_columns)})", agent_columns + map_columns),
        (f"everything ({len(X_learn.columns)})", list(X_learn.columns)),
    ]
}


def score_across_folds(columns, C=1.0):
    """Average accuracy over the five folds, using only these columns."""
    per_fold = []
    for learn_rows, check_rows in season_folds(learn):
        model = LogisticRegression(C=C, max_iter=5000)
        model.fit(X_learn.loc[learn_rows, columns], y_learn.loc[learn_rows])
        per_fold.append(model.score(X_learn.loc[check_rows, columns], y_learn.loc[check_rows]))
    return np.array(per_fold)


comparison = []
for name, columns in groups.items():
    folds_scored = score_across_folds(columns)
    comparison.append({
        "features": name,
        "mean %": round(folds_scored.mean() * 100, 1),
        "worst fold": round(folds_scored.min() * 100, 1),
        "best fold": round(folds_scored.max() * 100, 1),
    })

print(pd.DataFrame(comparison).to_string(index=False))
print(f"\nmap-picker baseline on the same folds: {results['map picker'].mean() * 100:.1f}%")

         features  mean %  worst fold  best fold
  picker only (1)    54.4        50.0       58.9
picker + map (11)    53.5        49.1       60.3
 agents only (27)    48.0        42.7       50.7
agents + map (37)    49.6        45.8       53.4
  everything (38)    51.4        47.3       53.7

map-picker baseline on the same folds: 55.5%


**Given only "who picked the map", the model scores 54.4%** — within about a point of the
rule itself. So it finds that signal perfectly well when nothing is in the way. Hand it the
other 37 columns and it falls to 51.4%. The agent features are not neutral; they cost
roughly three points.

(The picker-only model lands slightly *below* the hand-written rule, at 54.4% against
55.5%. That is the decider maps again: the rule guesses team A on them, while the model
leans on whatever its intercept has settled at.)

The agents-only row is the one that settles it: **48.0%, and it fails to clear 50.7% on any
of the five folds.** One fold below a coin flip is luck. Five in a row is a pattern.

Being precise about what that does *not* mean: it is not evidence that agent picks are
actively bad, or that choosing a particular agent makes a team lose. It means the model
finds patterns in its training maps that do not survive into the next chunk of the season.
With 27 agent columns and as few as 127 training maps in the first fold, there is ample
room to find coincidences. Applying those coincidences to fresh matches is worse than
useless, because the model is now confidently wrong where before it was merely guessing.

## 5. Does turning the regularisation up rescue it?

If the problem is 27 noisy columns shouting over one useful one, then squeezing the
coefficients harder might help — the noise has nothing real holding it up, so it should
shrink faster than the genuine signal does.

`C` controls that squeeze, backwards: **small `C` means a hard squeeze**, large `C` lets the
coefficients run free. The default is `C=1.0`. Alongside each score below is the average
size of the 27 agent coefficients and the size of the picker coefficient, which is what
shows whether the squeeze is landing where we hoped.

In [6]:
sweep = []
for C in [0.001, 0.01, 0.03, 0.1, 0.3, 1.0, 10.0, 100.0]:
    folds_scored, agent_sizes, picker_sizes = [], [], []
    for learn_rows, check_rows in season_folds(learn):
        model = LogisticRegression(C=C, max_iter=5000)
        model.fit(X_learn.loc[learn_rows], y_learn.loc[learn_rows])
        folds_scored.append(model.score(X_learn.loc[check_rows], y_learn.loc[check_rows]))

        coefficients = pd.Series(model.coef_[0], index=X_learn.columns)
        agent_sizes.append(coefficients[agent_columns].abs().mean())
        picker_sizes.append(abs(coefficients["map_picked_by_a"]))

    sweep.append({
        "C": C,
        "mean %": round(np.mean(folds_scored) * 100, 1),
        "worst": round(min(folds_scored) * 100, 1),
        "best": round(max(folds_scored) * 100, 1),
        "avg |agent coef|": round(np.mean(agent_sizes), 3),
        "|picker coef|": round(np.mean(picker_sizes), 3),
    })

print(pd.DataFrame(sweep).to_string(index=False))
print(f"\npicker-only model: {score_across_folds(picker_column).mean() * 100:.1f}%"
      f"      map-picker baseline: {results['map picker'].mean() * 100:.1f}%")

      C  mean %  worst  best  avg |agent coef|  |picker coef|
  0.001    49.6   45.7  54.8             0.002          0.015
  0.010    52.6   49.1  55.7             0.019          0.086
  0.030    53.5   47.8  56.6             0.045          0.139
  0.100    51.4   48.1  55.6             0.099          0.183
  0.300    51.2   48.1  54.8             0.169          0.202
  1.000    51.4   47.3  53.7             0.249          0.209
 10.000    51.6   47.3  53.4             0.343          0.202
100.000    51.3   47.3  54.3             0.368          0.198

picker-only model: 54.4%      map-picker baseline: 55.5%


Squeezing harder does help — 51.4% at the default rises to 53.5% at `C=0.03` — but it
never reaches the 54.4% you get by dropping the agent columns altogether.

The two coefficient columns show why. At the default `C=1.0` the *average* agent
coefficient (0.249) is **larger** than the picker coefficient (0.209): the model is
weighting each of 27 noise columns more heavily than the one column that carries signal.
Squeeze harder and the agents do shrink faster — at `C=0.01` the picker coefficient is about
4.5 times the average agent — which is exactly why the score climbs. But the squeeze cannot
shrink the agents to nothing without flattening the picker too, and at `C=0.001` everything
is crushed and the score falls back to a coin flip.

**One caution about that 53.5%.** We chose `C=0.03` by trying eight values and keeping the
best one, and 53.5 against 51.4 sits well inside the ±4-to-±6 point margin from section 2.
So the claim "0.03 is the right setting" is not one this evidence supports — try eight
things on a noisy measurement and one of them will look best whether or not it is.

The claim that does survive is the broader one: *no setting of `C` lets the agent features
beat the model that does not have them.* That holds at every value tried, which is a
pattern rather than a single measurement.

## What this notebook found

1. **A model built on pre-match agent picks does not predict the winner.** It scores 51.4%
   against 55.5% for a rule that guesses whoever chose the map (and team A when nobody
   did).
2. **The agent features actively cost accuracy.** Removing them raises the score to 54.4%.
   On their own they manage 48.0% and never clear 50.7% on any fold.
3. **Tuning does not fix it.** Regularisation recovers part of the gap and no more.
4. **This is the expected result, not a failure.** Notebook 03 predicted it: no single
   agent's presence predicted winning, so a model made mostly of agent columns has little to
   work with. A high score here would have meant a bug.

This is the "excluding player stats" model from the project plan — deliberately the weaker
of the two. It exists to be the thing the next model is measured against. The next notebook
adds each player's form going into the match, as a rolling average over their *earlier*
matches only, and the comparison between the two is the actual result.

****The test half has still not been touched.**

### One thing to fix before the next notebook

`build_features` builds its map columns from whichever maps appear in the data handed to
it. The map pool rotated almost exactly at our split — Fracture, Pearl and Split were last
played between 18 May and 22 June, and Corrode first appeared on 3 July. So the two halves
of the season do not produce the same columns:

| | columns |
|---|---|
| full season | 39 |
| learning half | 38 *(no Corrode)* |
| test half | 36 *(no Fracture, Pearl or Split)* |

A model fitted on the learning half therefore cannot score the test half as things stand —
the columns do not line up, and Corrode is a map it has never seen. `build_features` needs
to take a fixed list of columns so every slice of the season produces the same shape.

Worth noting this is not only a nuisance: the map pool changing mid-season is a sharper
demonstration of why the split has to be chronological than the agent data was. It is not
just that teams changed what they picked — the game changed what there was to pick.**